In [ ]:
from transformers import GPT2LMHeadModel

In [ ]:
model_hf  = GPT2LMHeadModel.from_pretrained("gpt2")
st_di     = model_hf.state_dict() # to get the raw tensors of the model

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
for k,v in st_di.items():
    print(k,v.shape)

transformer.wte.weight torch.Size([50257, 768])
transformer.wpe.weight torch.Size([1024, 768])
transformer.h.0.ln_1.weight torch.Size([768])
transformer.h.0.ln_1.bias torch.Size([768])
transformer.h.0.attn.c_attn.weight torch.Size([768, 2304])
transformer.h.0.attn.c_attn.bias torch.Size([2304])
transformer.h.0.attn.c_proj.weight torch.Size([768, 768])
transformer.h.0.attn.c_proj.bias torch.Size([768])
transformer.h.0.ln_2.weight torch.Size([768])
transformer.h.0.ln_2.bias torch.Size([768])
transformer.h.0.mlp.c_fc.weight torch.Size([768, 3072])
transformer.h.0.mlp.c_fc.bias torch.Size([3072])
transformer.h.0.mlp.c_proj.weight torch.Size([3072, 768])
transformer.h.0.mlp.c_proj.bias torch.Size([768])
transformer.h.1.ln_1.weight torch.Size([768])
transformer.h.1.ln_1.bias torch.Size([768])
transformer.h.1.attn.c_attn.weight torch.Size([768, 2304])
transformer.h.1.attn.c_attn.bias torch.Size([2304])
transformer.h.1.attn.c_proj.weight torch.Size([768, 768])
transformer.h.1.attn.c_proj.bias 

In [ ]:
(st_di['transformer.wte.weight'] == st_di['lm_head.weight']).all()

tensor(True)

In [ ]:
## if you check the data pointer of both matrix, its points to a single data point

print(st_di['transformer.wte.weight'].data_ptr()),
print(st_di['lm_head.weight'].data_ptr())

139145662418899
139145662418899


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
st_di["transformer.h.3.attn.c_attn.weight"].shape

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.imshow(st_di['transformer.wpe.weight'],cmap="gray");

In [ ]:
plt.plot(st_di['transformer.wpe.weight'][:,0:50]);

In [ ]:
# @title Model

import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

@dataclass
class GPTConfig:
    block_size  = 1024  # ==> block size
    vocab_size  = 50257 # ==> numbers of tokens ==> 50000 merges + 256 bytes token + 1 special token <|endoftext|>
    n_layer     = 12
    n_head      = 12
    n_emb       = 768   # ==> embedding dim

In [ ]:
class CasualSelfAttention(nn.Module):
    def __init__(self,config):
        super().__init__()
        assert config.n_emb % config.n_head ==0
        self.c_attn = nn.Linear(config.n_emb,3*config.n_emb)    # combined attention==> key,query,value projection for all heads,but in a batch
        self.c_proj = nn.Linear(config.n_emb,config.n_emb)      # output projection
        self.c_proj.NANOGPT_SCALE_INIT  = 1

        self.n_head = config.n_head
        self.n_emb  = config.n_emb

    def forward(self,x):
        B,T,C       = x.shape # batch_size, sequence length, embedding dim
        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        # nh                    = "number of heads",
        # hs                    = "head size"
        # C (number of channels)= nh * hs
        # e.g. in GPT-2 (124M)==> n_head    =12,
        #                         hs        =64, ==> nh * hs = C = 768 channels in the Transformer
        qkv     = self.c_attn(x)                                                    # B,T,3*n_emb
        q,k,v   = qkv.split(self.n_emb,dim=2)                                       # B,T,n_emb
        q       = q.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        k       = k.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        v       = v.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        y       = F.scaled_dot_product_attention(q,k,v,is_causal=True) #(B,n_head,T,head_size) # Flash attention==> its faster,cleaner,and scale better than Head object that created (reference link:-https://github.com/Vampaxx/Language_Model/blob/main/GPT_from_scratch/07_Adding_new_parameters.py )
        y       = y.transpose(1,2).contiguous().view(B,T,C)
        #output projection
        y       = self.c_proj(y)
        return y

In [ ]:
class MLP(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_emb,4*config.n_emb)
        self.gelu   = nn.GELU(approximate="tanh")   # there is no reason to use this approximation in nowdays, the time they develop this approximation they faced speed issue. thats why developed approximation
        self.c_proj = nn.Linear(config.n_emb * 4,config.n_emb)
        self.c_proj.NANOGPT_SCALE_INIT  = 1
    def forward(self,x):
        x   = self.c_fc(x)
        x   = self.gelu(x)
        x   = self.c_proj(x)
        return x

In [ ]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1   = nn.LayerNorm(config.n_emb)
        self.attn   = CasualSelfAttention(config)
        self.ln_2   = nn.LayerNorm(config.n_emb)
        self.mlp    = MLP(config)

    def forward(self,x):
        x   = x + self.attn(self.ln_1(x))
        x   = x + self.mlp(self.ln_2(x))
        return x

In [ ]:
model_type  = "gpt2"
config_args = {
    "gpt2"          : dict(n_layer  = 12, n_head = 12,n_emb = 768),
    "gpt2-medium"   : dict(n_layer  = 24, n_head = 16,n_emb = 1024),
    "gpt2-large"    : dict(n_layer  = 36, n_head = 20,n_emb = 1280),
    "gpt2-xl"       : dict(n_layer  = 48, n_head = 25,n_emb = 1600)
}[model_type]
config_args['vocab_size']   = 50257
config_args['block_size']   = 1024

In [ ]:
config_args

In [ ]:

@dataclass
class GPTConfig:
    block_size:int  = 1024  # ==> block size
    vocab_size:int  = 50257 # ==> numbers of tokens ==> 50000 merges + 256 bytes token + 1 special token <|endoftext|>
    n_layer:int     = 12
    n_head:int      = 12
    n_emb:int       = 768   # ==> embedding dim

In [ ]:
GPTConfig(**config_args)

In [ ]:
class GPT(nn.Module):
  def __init__(self,config):
      super().__init__()
      self.config = config
      self.transformer = nn.ModuleDict(dict(
          wte     = nn.Embedding(config.vocab_size,config.n_emb),           # token embeding
          wpe     = nn.Embedding(config.block_size,config.n_emb),           # position embedding
          h       = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),    # self attention heads
          ln_f    = nn.LayerNorm(config.n_emb)
      ))
      self.lm_head= nn.Linear(config.n_emb,config.vocab_size,bias=False)    # lm_head is following be softmax, and bias not make any sence or improvement in learning.
      # The bias term in this case would just add a constant to each token’s logit — this doesn’t meaningfully improve learning,
  def forward(self,idx):
    # shape of idx is (B,T)
    B,T     = idx.shape
    assert T<=self.config.block_size, f"cannot forward sequence of length {T},block_size is only {self.config.block_size}"
    pos     = torch.arange(0,T,dtype=torch.long,device=idx.device)  # shape (T)
    pos_emb = self.transformer.wpe(pos)                             # position embedding of shape (_,T,n_emb)
    tok_emb = self.transformer.wte(idx)                             # token embedding of shape    (B,T,n_emb)

    x       = tok_emb + pos_emb   # (B,T,n_emb)
    for block in self.transformer.h:
      x = block(x)
    #forward the final layerorm and classifier
    x       = self.transformer.ln_f(x)
    logits  = self.lm_head(x)
    return logits

  @classmethod
  def from_pretrained(cls,model_type):
    """Loads the pre-trained GPT-2 model weight from huggingface"""
    assert model_type in {"gpt2","gpt2-medium","gpt2-large","gpt2-xl"}
    from transformers import GPT2LMHeadModel
    print("Loading weights from pretrained gpt: %s"% model_type)

    config_args = {
        "gpt2"          : dict(n_layer  = 12, n_head = 12,n_emb = 768),   # 124M
        "gpt2-medium"   : dict(n_layer  = 24, n_head = 16,n_emb = 1024),  # 350M
        "gpt2-large"    : dict(n_layer  = 36, n_head = 20,n_emb = 1280),  # 774M
        "gpt2-xl"       : dict(n_layer  = 48, n_head = 25,n_emb = 1600),   # 1558M
    }[model_type]
    config_args['vocab_size']   = 50257
    config_args['block_size']   = 1024
    config                      = GPTConfig(**config_args)
    model                       = GPT(config)
    sd                          = model.state_dict()
    sd_keys                     = sd.keys()
    sd_keys                     = [k for k in sd_keys if not k.endswith(".attn.bias")]  # avoiding mask / buffers, it's not a parameters

    model_hf                    = GPT2LMHeadModel.from_pretrained(model_type)
    sd_hf                       = model_hf.state_dict()
    # copying while ensuring all the parameters are alligned and match in name and shapes
    sd_keys_hf    = sd_hf.keys()
    sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
    sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')]        # same, just the mask (buffer)
    transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
    # basically the openai checkpoints use a "Conv1D" module, but we only want to use a vanilla Linear
    # this means that we have to transpose these weights when we import them
    assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
    for k in sd_keys_hf:
        if any(k.endswith(w) for w in transposed):
            # special treatment for the Conv1D weights we need to transpose
            assert sd_hf[k].shape[::-1] == sd[k].shape
            with torch.no_grad():
                sd[k].copy_(sd_hf[k].t())
        else:
            # vanilla copy over the other parameters
            assert sd_hf[k].shape == sd[k].shape
            with torch.no_grad():
                sd[k].copy_(sd_hf[k])

    return model

In [ ]:
model = GPT.from_pretrained("gpt2")
print("Its works")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"The available device is {device}")

In [ ]:
num_return_sequences  = 5
max_length            = 30

In [ ]:
model = GPT.from_pretrained("gpt2")
model.eval()    # used becasuse we are not going to change the parameter
model.to(device)

In [ ]:
import time
import tiktoken

In [ ]:
enc     = tiktoken.get_encoding("gpt2")
tokens  = enc.encode("Hello, I'm an Large languag model")
tokens  = torch.tensor(tokens,dtype=torch.long)
tokens  = tokens.unsqueeze(0).repeat(num_return_sequences,1)

x = tokens.to(device)

In [ ]:
x.shape

In [ ]:
model(x).shape

In [ ]:
while x.size(1) < 30: # max_length=30
  # forward the model to get the logits
  with torch.no_grad():
      logits = model(x) # (B, T, vocab_size)
      # take the logits at the last position
      logits = logits[:, -1, :] # (B, vocab_size)
      # get the probabilities
      probs = F.softmax(logits, dim=-1)
      # do top-k sampling of 50 (huggingface pipeline default)
      # topk_probs here becomes (5, 50), topk_indices is (5, 50)
      topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)
      # select a token from the top-k probabilities
      # note: multinomial does not demand the input to sum to 1
      ix = torch.multinomial(topk_probs, 1) # (B, 1)
      # gather the corresponding indices
      xcol = torch.gather(topk_indices, -1, ix) # (B, 1)
      # append to the sequence
      x = torch.cat((x, xcol), dim=1)

In [ ]:
for i in range(num_return_sequences):
  print(">> ",enc.decode(x[i].tolist()))

In [ ]:
# @title sampling with Argmax
torch.manual_seed(42)
torch.cuda.manual_seed(42)
while x.size(1) < 30:# max_length = 30
  with torch.no_grad():
    logits = model(x)  # (B, T, vocab_size)
    logits = logits[:, -1, :]  # take last token's logits => (B, vocab_size)
    probs  = F.softmax(logits, dim=-1)  # (B, vocab_size), optional (not needed for argmax)

    # Use argmax to get the most likely token (greedy decoding)
    xcol = torch.argmax(probs, dim=-1, keepdim=True)  # shape (B, 1)

    # Append to sequence
    x = torch.cat((x, xcol), dim=1)

In [ ]:
enc.decode(x[0].tolist())

In [ ]:
for i in range(num_return_sequences):
  print(">> ",enc.decode(x[i].tolist()))


## Lets do Sampling loop with random initialized Model

In [ ]:
model = GPT(GPTConfig())
model.eval()
model.to(device)

enc     = tiktoken.get_encoding("gpt2")
tokens  = enc.encode("Hello, I'm an Large languag model")
tokens  = torch.tensor(tokens,dtype=torch.long)
tokens  = tokens.unsqueeze(0).repeat(num_return_sequences,1)

x = tokens.to(device)

while x.size(1) < 30: # max_length=30
  # forward the model to get the logits
  with torch.no_grad():
      logits = model(x) # (B, T, vocab_size)
      # take the logits at the last position
      logits = logits[:, -1, :] # (B, vocab_size)
      # get the probabilities
      probs = F.softmax(logits, dim=-1)
      # do top-k sampling of 50 (huggingface pipeline default)
      # topk_probs here becomes (5, 50), topk_indices is (5, 50)
      topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)
      # select a token from the top-k probabilities
      # note: multinomial does not demand the input to sum to 1
      ix    = torch.multinomial(topk_probs, 1) # (B, 1)
      # gather the corresponding indices
      xcol  = torch.gather(topk_indices, -1, ix) # (B, 1)
      # append to the sequence
      x     = torch.cat((x, xcol), dim=1)



for i in range(num_return_sequences):
  print(">> ",enc.decode(x[i].tolist()))

# Layer normalization & Batch Normalization

- BatchNorm = column-wise (per feature, across batch)
- LayerNorm = row-wise (per token/sequence, across feature)

In [ ]:
model